# 🔍 Diagnóstico: Vazamento de Dados no Modelo Random Forest

Notebook pra apresentar pro time. Reúne, em ordem, todas as queries usadas pra
investigar por que o `03_modelo_random_forest.ipynb` chega a 98,3% de acurácia — e mostra
que a maior parte dessa performance vem de colunas que são versões diretas ou indiretas
do próprio resultado do ano, não de características independentes do município.

**Como usar na reunião:** vai rodando célula por célula (`Shift+Enter`), lendo o texto
antes de cada uma — cada bloco prova um ponto específico.

## Roteiro
1. Carregar os dados
2. `media_portugues` é quase o próprio resultado
3. As proporções por nível de proficiência são ainda mais diretas
4. As metas do programa embutem a trajetória histórica do município
5. Resumo: quais das 24 features do Modelo A têm vazamento
6. Quanto sinal sobra sem essas colunas (o motivo do Modelo D)


## 1. Carregar os dados

In [1]:
import pandas as pd
import duckdb
from pathlib import Path

GOLD_PATH = Path('../data/gold/gold_indicador_municipio.parquet')
df = pd.read_parquet(GOLD_PATH)
print(f'Dataset carregado: {df.shape[0]} registros, {df.shape[1]} colunas')


Dataset carregado: 15238 registros, 30 colunas


## 2. `media_portugues` é quase o próprio resultado

`media_portugues` é a nota média do município na mesma prova (mesmo ano) que gera a
`taxa_alfabetizacao`. Granularidade: 1 linha por (município, ano, rede).

In [2]:
duckdb.sql("""
    SELECT id_municipio, ano, rede_label, media_portugues, taxa_alfabetizacao
    FROM df
    WHERE id_municipio = 1100031
    ORDER BY ano
""").df()


,id_municipio,ano,rede_label,media_portugues,taxa_alfabetizacao
0,1100031,2023,Estadual,767.8763,69.10
1,1100031,2024,Estadual,759.6600,75.88


In [3]:
duckdb.sql("SELECT corr(media_portugues, taxa_alfabetizacao) AS correlacao FROM df").df()
# 0,926 — correlação altíssima, porque vem da mesma edição da prova


,correlacao
0,0.925939


## 3. As proporções por nível de proficiência são ainda mais diretas

`proporcao_aluno_nivel_0` a `_8` são a distribuição dos alunos por nível na mesma prova.
Somando os níveis 5 a 8, praticamente reconstruímos a `taxa_alfabetizacao`.

In [4]:
cols = ", ".join([f"proporcao_aluno_nivel_{n}" for n in range(9)])
duckdb.sql(f"""
    SELECT ano, taxa_alfabetizacao, {cols}
    FROM df
    WHERE id_municipio = 1100189
    ORDER BY ano
""").df().T


,0,1
ano,2023.00,2024.00
taxa_alfabetizacao,69.73,81.06
proporcao_aluno_nivel_0,NaN,1.31
proporcao_aluno_nivel_1,NaN,1.27
proporcao_aluno_nivel_2,NaN,2.75
proporcao_aluno_nivel_3,NaN,4.99
proporcao_aluno_nivel_4,NaN,14.96
proporcao_aluno_nivel_5,NaN,42.45
proporcao_aluno_nivel_6,NaN,24.81
proporcao_aluno_nivel_7,NaN,5.61


In [5]:
# Testando a soma dos níveis 5..8 contra a taxa real, pra todo o dataset
sub = df.dropna(subset=[f'proporcao_aluno_nivel_{n}' for n in range(9)] + ['taxa_alfabetizacao'])
soma_5_8 = sub[[f'proporcao_aluno_nivel_{n}' for n in range(5, 9)]].sum(axis=1)

import numpy as np
correlacao = np.corrcoef(soma_5_8, sub['taxa_alfabetizacao'])[0, 1]
erro_medio = (soma_5_8 - sub['taxa_alfabetizacao']).abs().mean()

print(f'Correlação soma(nível 5..8) x taxa_alfabetizacao: {correlacao:.3f}')
print(f'Erro médio: {erro_medio:.1f} pontos percentuais')
print(f'(baseado em {len(sub)} linhas com essas colunas preenchidas)')


Correlação soma(nível 5..8) x taxa_alfabetizacao: 0.983
Erro médio: 5.9 pontos percentuais
(baseado em 7553 linhas com essas colunas preenchidas)


## 4. As metas do programa embutem a trajetória histórica do município

`meta_2024`...`meta_2030` vêm de `meta_alfabetizacao_municipio.csv`: uma trajetória
individualizada por município, calculada a partir da sua própria taxa-base
(`taxa_meta_base`) até chegar a 80% em 2030. Por isso correlacionam tão forte com o
resultado real — não são "independentes" do município, são derivadas dele.

In [6]:
duckdb.sql("""
    SELECT
        corr(taxa_meta_base, taxa_alfabetizacao) AS corr_taxa_meta_base,
        corr(meta_2024, taxa_alfabetizacao)      AS corr_meta_2024,
        corr(meta_2025, taxa_alfabetizacao)      AS corr_meta_2025,
        corr(meta_2026, taxa_alfabetizacao)      AS corr_meta_2026,
        corr(meta_2027, taxa_alfabetizacao)      AS corr_meta_2027,
        corr(meta_2028, taxa_alfabetizacao)      AS corr_meta_2028,
        corr(meta_2029, taxa_alfabetizacao)      AS corr_meta_2029
    FROM df
""").df().T.rename(columns={0: 'correlacao'})


,correlacao
corr_taxa_meta_base,0.924029
corr_meta_2024,0.737612
corr_meta_2025,0.733831
corr_meta_2026,0.731816
corr_meta_2027,0.730795
corr_meta_2028,0.731416
corr_meta_2029,0.733458


## 5. Resumo: quais das 24 features do Modelo A têm vazamento

Juntando os três diagnósticos acima, dá pra classificar as 24 features que o Modelo A
usa em 3 grupos por risco de vazamento.

In [7]:
resumo = pd.DataFrame([
    {'grupo': 'Mesma prova / mesmo ano (vazamento direto)',
     'colunas': 'media_portugues, proporcao_aluno_nivel_0..8',
     'n_colunas': 10, 'risco': 'Alto — é o próprio resultado fatiado'},
    {'grupo': 'Metas do programa (vazamento indireto)',
     'colunas': 'taxa_meta_base, meta_2024..meta_2030, meta_ano_vigente',
     'n_colunas': 9, 'risco': 'Alto — é a trajetória do próprio município'},
    {'grupo': 'Estruturais (seguras)',
     'colunas': 'ano, serie, rede, rede_label, percentual_participacao',
     'n_colunas': 5, 'risco': 'Baixo — características reais do município-ano'},
])
print(f"Total: {resumo['n_colunas'].sum()} colunas (bate com as 24 features do Modelo A)")
resumo


Total: 24 colunas (bate com as 24 features do Modelo A)


,grupo,colunas,n_colunas,risco
0,Mesma prova / mesmo ano (vazamento direto),"media_portugues, proporcao_aluno_nivel_0..8",10,Alto — é o próprio resultado fatiado
1,Metas do programa (vazamento indireto),"taxa_meta_base, meta_2024..meta_2030, meta_ano...",9,Alto — é a trajetória do próprio município
2,Estruturais (seguras),"ano, serie, rede, rede_label, percentual_parti...",5,Baixo — características reais do município-ano


## 6. Quanto sinal sobra sem essas colunas

Retreinando o mesmo Random Forest só com as 5 colunas "estruturais" (grupo seguro):

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

df_c = df.copy()
df_c['target'] = (df_c['taxa_alfabetizacao'] >= 50).astype(int)

FEATURE_COLS_C = ['ano', 'serie', 'rede', 'rede_label', 'percentual_participacao']
CAT_COLS = ['rede', 'rede_label', 'serie']
NUM_COLS_C = [c for c in FEATURE_COLS_C if c not in CAT_COLS]

X = df_c[FEATURE_COLS_C]
y = df_c['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), NUM_COLS_C),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                       ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT_COLS),
])
modelo_estrutural = Pipeline([('preprocessor', preprocessor),
                               ('model', RandomForestClassifier(n_estimators=200, max_depth=10,
                                                                  min_samples_leaf=5, random_state=42, n_jobs=-1))])
modelo_estrutural.fit(X_train, y_train)
y_pred = modelo_estrutural.predict(X_test)
y_proba = modelo_estrutural.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.1%}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_proba):.3f}")
print(f"Baseline (sempre prever a classe majoritária): {max(y_test.mean(), 1-y_test.mean()):.1%}")
print()
print('Conclusão: sem as colunas de vazamento, o modelo praticamente empata com o baseline ingênuo —')
print('ou seja, o dataset atual quase não tem sinal preditivo genuíno fora dessas colunas.')
print()
print('A versão corrigida (usando dado do ano anterior pra prever o ano seguinte, sem vazamento)')
print('está no notebook 03b_modelo_defasagem_temporal.ipynb — Accuracy 83,6%, AUC 0,840.')


Accuracy: 72.7%
AUC-ROC:  0.624
Baseline (sempre prever a classe majoritária): 72.1%

Conclusão: sem as colunas de vazamento, o modelo praticamente empata com o baseline ingênuo —
ou seja, o dataset atual quase não tem sinal preditivo genuíno fora dessas colunas.

A versão corrigida (usando dado do ano anterior pra prever o ano seguinte, sem vazamento)
está no notebook 03b_modelo_defasagem_temporal.ipynb — Accuracy 83,6%, AUC 0,840.
